# Bước 3: Data preprocessing – tiền xử lý dữ liệu

Bước 2 cho ta biết dữ liệu trông thế nào. Bước 3 biến dữ liệu thô thành bảng mà mô hình học được.
Mỗi quyết định ở đây đều dựa trên một bằng chứng từ bước 2.

| Việc | Hàm | Nằm ở |
|---|---|---|
| Làm sạch | `clean_data` | `pipelines/data_processing/nodes.py` |
| Tạo và chọn feature | `create_features` | `pipelines/feature_engineering/nodes.py` |
| Chia train/test | `split_data` | `pipelines/feature_engineering/nodes.py` |

Notebook này chạy các hàm đó **trong bộ nhớ** để bạn nhìn từng bước. Chạy thật và lưu file thì dùng `kedro run`.

In [ ]:
import importlib.util
import logging
import sys
import warnings
from pathlib import Path

# Kiểm tra kernel đã có đủ thư viện chưa, để báo lỗi dễ hiểu nếu thiếu
required = ["kedro", "kedro_datasets", "pandas", "pyarrow", "scipy", "sklearn", "matplotlib"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ModuleNotFoundError(
        f"Kernel này ({sys.executable}) thiếu: {missing}.\n"
        "Chạy trong terminal rồi khởi động lại kernel:\n"
        f'  "{sys.executable}" -m pip install -r requirements.txt'
    )

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project


def find_project_root() -> Path:
    # Tìm thư mục gốc của dự án Kedro, dù notebook được mở từ đâu
    here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "x")).resolve().parent
    for folder in (here, *here.parents, Path.cwd(), *Path.cwd().parents):
        if (folder / "conf").is_dir() and (folder / "pyproject.toml").exists():
            return folder
    for pyproject in Path.cwd().glob("*/pyproject.toml"):  # mở từ thư mục cha
        if "[tool.kedro]" in pyproject.read_text(encoding="utf-8"):
            return pyproject.parent
    raise FileNotFoundError("Không tìm thấy dự án Kedro (thư mục có conf/ và pyproject.toml)")


project_path = find_project_root()
bootstrap_project(project_path)  # thêm thư mục src vào đường dẫn import
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    catalog = context.catalog  # dữ liệu khai báo trong conf/base/catalog.yml
    params = context.params    # tham số trong conf/base/parameters*.yml

for name in ("kedro", "netflix"):  # bớt log của Kedro trong notebook
    logging.getLogger(name).setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Python đang dùng:", sys.executable)
print("Thư mục dự án:", project_path)

In [ ]:
from netflix.pipelines.data_processing.nodes import clean_data
from netflix.pipelines.feature_engineering.nodes import create_features, split_data
from netflix.pipelines.statistics.nodes import target_association

raw = catalog.load("netflix_raw")
print("Dữ liệu gốc:", raw.shape)

## 3.1 Làm sạch
`clean_data` bỏ khách trùng `customer_id`, dòng thiếu giá trị, và giá trị vô lý (tuổi hoặc phí ≤ 0, giờ xem âm), rồi ép kiểu.
Bước 2 cho thấy dữ liệu này đã sạch, nên kỳ vọng là **không mất dòng nào**. Nhưng hàm vẫn cần có để bảo vệ dữ liệu mới sau này.

In [ ]:
clean = clean_data(raw, params["data_processing"])
print(f"Trước: {len(raw)} dòng | Sau: {len(clean)} dòng | Bị loại: {len(raw) - len(clean)}")
clean.dtypes

## 3.2 Bỏ cột nào, giữ cột nào?
Bằng chứng từ bước 2: kiểm định thống kê giữa từng cột và `churned`.
Cột có `significant = False` (p-value ≥ 0,05) gần như không giúp dự đoán.

In [ ]:
association = target_association(raw, params["statistics"])
association[["feature", "type", "strength", "p_value", "significant"]]

In [ ]:
fe = params["feature_engineering"]
no_signal = set(association.loc[~association["significant"], "feature"])
print("Cột không liên quan theo thống kê:", sorted(no_signal))
print("Cột bị bỏ trong parameters_feature_engineering.yml:", sorted(fe["drop_columns"]))
print("Cột không liên quan mà vẫn giữ lại:", sorted(no_signal - set(fe["drop_columns"])) or "không có")

`customer_id` cũng bị bỏ dù không nằm trong bảng kiểm định: nó chỉ là mã định danh, mô hình mà học theo nó thì chỉ là học thuộc lòng.

> Bỏ cột là quyết định **có thể sai**: cột yếu riêng lẻ đôi khi có ích khi kết hợp với cột khác. Muốn kiểm chứng, bạn xoá bớt tên trong `drop_columns`, chạy lại notebook 05 và so điểm.

## 3.3 Biến đổi cột lệch (log)
`watch_hours` và `avg_watch_time_per_day` lệch phải rất mạnh (skew 2,3 và 15,8): đa số khách xem ít, vài khách xem cực nhiều.
`log1p(x) = log(1 + x)` nén đuôi dài lại để các giá trị cực đoan không lấn át mô hình tuyến tính.

In [ ]:
cols = fe["log_columns"]
pd.DataFrame({"skew_truoc": clean[cols].skew(), "skew_sau_log1p": np.log1p(clean[cols]).skew()}).round(2)

## 3.4 Tạo feature mới từ điều quan sát được
Bước 2 cho thấy hai quan hệ **không đi theo đường thẳng**, mô hình tuyến tính khó tự bắt được:
- `last_login_days`: tỉ lệ rời tăng vọt sau khoảng 30 ngày không đăng nhập → feature `is_inactive`.
- `number_of_profiles`: tỉ lệ rời giảm mạnh từ 4 hồ sơ → feature `has_many_profiles`.

Ngưỡng 30 và 4 nằm trong `parameters_feature_engineering.yml`, bạn đổi được.

In [ ]:
features = create_features(clean, fe)
for flag in ["is_inactive", "has_many_profiles"]:
    print(flag)
    print(features.groupby(flag)["churned"].agg(so_khach="count", ti_le_roi="mean").round(3), "\n")

In [ ]:
print("Bảng feature cuối cùng:", features.shape)
features.head()

## 3.5 Chia train/test
Chia **trước** khi huấn luyện, và có `stratify` để hai tập có cùng tỉ lệ rời bỏ.
Tập test để dành, không đụng tới cho đến bước 6.

In [ ]:
X_train, X_test, y_train, y_test = split_data(features, fe, params["model_options"])
print("train:", X_train.shape, "| test:", X_test.shape)
print("Tỉ lệ rời - train:", round(y_train["churned"].mean(), 3), "| test:", round(y_test["churned"].mean(), 3))
print("Không trùng dòng giữa train và test:", set(X_train.index).isdisjoint(X_test.index))

## Vì sao chưa mã hoá chữ thành số và chuẩn hoá?
One-hot và chuẩn hoá (trừ trung bình, chia độ lệch) **học các con số từ dữ liệu**. Nếu làm trên cả bảng trước khi chia, thông tin của tập test rò vào tập train (*data leakage*) và điểm số đẹp hơn thực tế.
Vì vậy hai việc đó nằm **bên trong** mô hình (`training/nodes.py`), nơi nó chỉ học từ phần train của mỗi lần kiểm tra chéo.

## Ghi chú của bạn
Viết vài dòng bằng lời của bạn: bạn học được gì, điều gì bất ngờ, bạn sẽ thử gì tiếp?

-
-
-